# Filtrado 8
Consiste en sacar las dwas, asociadas a tareas y ocupaciones. Paso 1: traduccion de ocupacion a dwas. Lista de dwas also. 

## SQL

In [5]:
# -*- coding: utf-8 -*-
from pathlib import Path

# Lista tal como la diste
ocupations = [
    'Industrial Production Managers','Quality Control Systems Managers','Supply Chain Managers',
    'Human Resources Managers','Logistics Engineers','Bioengineers and Biomedical Engineers',
    'Chemical Engineers','Validation Engineers','Manufacturing Engineers','Mechanical Engineers',
    'Automotive Engineers','Mechatronics Engineers','Robotics Engineers','Robotics Technicians',
    'Industrial Engineering Technologists and Technicians',
    'Mechanical Engineering Technologists and Technicians',
    'Biochemists and Biophysicists','Microbiologists','Chemists'
]


In [6]:

# Carpeta de salida (cámbiala si quieres)
OUT_DIR = Path("filtrado8")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TEMPLATE = """USE onet;

SELECT
  o.onetsoc_code,
  o.title,
  td.task_id,
  t.task,
  td.dwa_id,
  dwaref.dwa_title
FROM tasks_to_dwas td
JOIN occupation_data o ON td.onetsoc_code = o.onetsoc_code
JOIN dwa_reference dwaref ON td.dwa_id = dwaref.dwa_id
JOIN task_statements t ON td.task_id = t.task_id
WHERE o.title = '{title_escaped}';
"""


In [7]:

for title in ocupations:
    # title=title.lower()
    # Escapar comillas simples para SQL (por si en el futuro aparece alguna)
    title_escaped = title.replace("'", "''")
    # Nombre de archivo: espacios -> _
    filename = title.replace(" ", "_") + ".sql"
    sql_text = TEMPLATE.format(title_escaped=title_escaped)
    (OUT_DIR / filename).write_text(sql_text, encoding="utf-8")

print(f"Generados {len(ocupations)} archivos .sql en: {OUT_DIR.resolve()}")


Generados 19 archivos .sql en: /Users/jpardo/Desktop/Proyectos/AI4LABOUR/Mejoras/eerr/filtrado8


## Agrupacion de todos los datasets

In [8]:
import os
import pandas as pd

path = "/Users/jpardo/Desktop/Proyectos/AI4LABOUR/mejoras/eerr/filtrado8"

# Listar solo CSV
archivos = [i for i in os.listdir(path) if i.endswith('.csv')]

dfs = []  # lista para acumular dataframes

for i in archivos:
    path_archivo = os.path.join(path, i)
    df_temp = pd.read_csv(path_archivo)
    df_temp["archivo_origen"] = i  # opcional: para saber de qué archivo viene cada fila
    dfs.append(df_temp)

# Unir todos los DataFrames
df_final = pd.concat(dfs, ignore_index=True)

print(f"Se han cargado {len(df_final)} filas de {len(archivos)} archivos.")


Se han cargado 560 filas de 19 archivos.


In [9]:
display(df_final)

,onetsoc_code,title,task_id,task,dwa_id,dwa_title,archivo_origen
0,17-2141.02,Automotive Engineers,16421,"Read current literature, attend meetings or co...",4.A.2.b.3.I01.D20,Update technical knowledge.,Automotive_Engineers.csv
1,17-2141.02,Automotive Engineers,16422,Establish production or quality control standa...,4.A.2.b.2.I07.D06,Determine operational criteria or specifications.,Automotive_Engineers.csv
2,17-2141.02,Automotive Engineers,16423,Prepare or present technical or project status...,4.A.3.b.6.I15.D04,Prepare operational reports.,Automotive_Engineers.csv
3,17-2141.02,Automotive Engineers,16424,Develop or implement operating methods or proc...,4.A.2.b.1.I09.D06,Implement design or process improvements.,Automotive_Engineers.csv
4,17-2141.02,Automotive Engineers,16424,Develop or implement operating methods or proc...,4.A.2.b.4.I03.D14,Develop technical methods or processes.,Automotive_Engineers.csv
...,...,...,...,...,...,...,...
555,11-3051.01,Quality Control Systems Managers,15428,Analyze quality control test results and provi...,4.A.2.a.4.I07.D12,Analyze data to inform operational decisions o...,Quality_Control_Systems_Managers.csv
556,11-3051.01,Quality Control Systems Managers,15428,Analyze quality control test results and provi...,4.A.4.a.2.I03.D14,Confer with organizational members to accompli...,Quality_Control_Systems_Managers.csv
557,11-3051.01,Quality Control Systems Managers,15429,"Oversee workers including supervisors, inspect...",4.A.4.b.4.I01.D13,Supervise employees.,Quality_Control_Systems_Managers.csv
558,11-3051.01,Quality Control Systems Managers,15430,Monitor performance of quality control systems...,4.A.1.a.2.I02.D05,Monitor organizational procedures to ensure pr...,Quality_Control_Systems_Managers.csv


In [10]:
!pip install openpyxl


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [11]:
# Cargar el Excel
ruta_excel = "/Users/jpardo/Desktop/Proyectos/AI4LABOUR/mejoras/eerr/Jobs List ICBE.xlsx"
df_excel = pd.read_excel(ruta_excel, sheet_name="ICBE")
# Pasar nombres de columnas a minúsculas
df_excel.columns = df_excel.columns.str.lower()


display(df_excel)

,o.net.soc.code,title,icbe,description
0,11-3051.00,Industrial Production Managers,Primary,"Plan, direct, or coordinate the work activitie..."
1,11-3051.01,Quality Control Systems Managers,Secondary,"Plan, direct, or coordinate quality assurance ..."
2,11-3071.04,Supply Chain Managers,Secondary,"Direct or coordinate production, purchasing, w..."
3,11-3121.00,Human Resources Managers,Secondary,"Plan, direct, or coordinate human resources ac..."
4,13-1081.01,Logistics Engineers,Secondary,Design or analyze operational solutions for pr...
5,17-2031.00,Bioengineers and Biomedical Engineers,NaN,"Apply knowledge of engineering, biology, chemi..."
6,17-2041.00,Chemical Engineers,Primary,Design chemical plant equipment and devise pro...
7,17-2112.02,Validation Engineers,NaN,Design or plan protocols for equipment or proc...
8,17-2112.03,Manufacturing Engineers,Primary,"Design, integrate, or improve manufacturing sy..."
9,17-2141.00,Mechanical Engineers,Primary,Perform engineering duties in planning and des...


In [12]:
# Unir DataFrames
# Aquí debes definir la columna clave común en ambos DF, por ejemplo 'dwa_id'
df_merged = pd.merge(df_final, df_excel, on="title", how="left")

print(f"DataFrame final tiene {df_merged.shape[0]} filas y {df_merged.shape[1]} columnas.")

DataFrame final tiene 560 filas y 10 columnas.


In [13]:
display(df_merged)

,onetsoc_code,title,task_id,task,dwa_id,dwa_title,archivo_origen,o.net.soc.code,icbe,description
0,17-2141.02,Automotive Engineers,16421,"Read current literature, attend meetings or co...",4.A.2.b.3.I01.D20,Update technical knowledge.,Automotive_Engineers.csv,17-2141.02,NaN,Develop new or improved designs for vehicle st...
1,17-2141.02,Automotive Engineers,16422,Establish production or quality control standa...,4.A.2.b.2.I07.D06,Determine operational criteria or specifications.,Automotive_Engineers.csv,17-2141.02,NaN,Develop new or improved designs for vehicle st...
2,17-2141.02,Automotive Engineers,16423,Prepare or present technical or project status...,4.A.3.b.6.I15.D04,Prepare operational reports.,Automotive_Engineers.csv,17-2141.02,NaN,Develop new or improved designs for vehicle st...
3,17-2141.02,Automotive Engineers,16424,Develop or implement operating methods or proc...,4.A.2.b.1.I09.D06,Implement design or process improvements.,Automotive_Engineers.csv,17-2141.02,NaN,Develop new or improved designs for vehicle st...
4,17-2141.02,Automotive Engineers,16424,Develop or implement operating methods or proc...,4.A.2.b.4.I03.D14,Develop technical methods or processes.,Automotive_Engineers.csv,17-2141.02,NaN,Develop new or improved designs for vehicle st...
...,...,...,...,...,...,...,...,...,...,...
555,11-3051.01,Quality Control Systems Managers,15428,Analyze quality control test results and provi...,4.A.2.a.4.I07.D12,Analyze data to inform operational decisions o...,Quality_Control_Systems_Managers.csv,11-3051.01,Secondary,"Plan, direct, or coordinate quality assurance ..."
556,11-3051.01,Quality Control Systems Managers,15428,Analyze quality control test results and provi...,4.A.4.a.2.I03.D14,Confer with organizational members to accompli...,Quality_Control_Systems_Managers.csv,11-3051.01,Secondary,"Plan, direct, or coordinate quality assurance ..."
557,11-3051.01,Quality Control Systems Managers,15429,"Oversee workers including supervisors, inspect...",4.A.4.b.4.I01.D13,Supervise employees.,Quality_Control_Systems_Managers.csv,11-3051.01,Secondary,"Plan, direct, or coordinate quality assurance ..."
558,11-3051.01,Quality Control Systems Managers,15430,Monitor performance of quality control systems...,4.A.1.a.2.I02.D05,Monitor organizational procedures to ensure pr...,Quality_Control_Systems_Managers.csv,11-3051.01,Secondary,"Plan, direct, or coordinate quality assurance ..."


### Ocuppations

In [14]:
from pprint import pprint
occupations = df_excel['title'].tolist()
pprint(occupations)

['Industrial Production Managers',
 'Quality Control Systems Managers',
 'Supply Chain Managers',
 'Human Resources Managers',
 'Logistics Engineers',
 'Bioengineers and Biomedical Engineers',
 'Chemical Engineers',
 'Validation Engineers',
 'Manufacturing Engineers',
 'Mechanical Engineers',
 'Automotive Engineers',
 'Mechatronics Engineers',
 'Robotics Engineers',
 'Robotics Technicians',
 'Industrial Engineering Technologists and Technicians',
 'Mechanical Engineering Technologists and Technicians',
 'Biochemists and Biophysicists',
 'Microbiologists',
 'Chemists']


In [15]:
print(len(occupations))

19


### DWAs

In [16]:
conteo = pd.DataFrame(df_final["dwa_title"].value_counts())

display(conteo)


,count
dwa_title,
Design industrial processing systems.,11
"Recommend technical design or process changes to improve efficiency, quality, or performance.",11
Program robotic equipment.,11
Design electromechanical equipment or systems.,11
Estimate operational costs.,10
...,...
Monitor performance of organizational members or partners.,1
Manage inventories of products or organizational resources.,1
Develop organizational goals or objectives.,1


In [17]:
num_compartidos = conteo[conteo['count'] > 1]
display(num_compartidos)

,count
dwa_title,
Design industrial processing systems.,11
"Recommend technical design or process changes to improve efficiency, quality, or performance.",11
Program robotic equipment.,11
Design electromechanical equipment or systems.,11
Estimate operational costs.,10
...,...
Direct operational or production activities.,2
Monitor processes for compliance with standards.,2
Research industrial processes or operations.,2


In [18]:
221-126

95

Hay 95 dwa que se comparten entre las 19 occupations. 

## DWAs save


In [26]:
dwa = pd.DataFrame(df_final['dwa_title'])
display(dwa)

,dwa_title
0,Update technical knowledge.
1,Determine operational criteria or specifications.
2,Prepare operational reports.
3,Implement design or process improvements.
4,Develop technical methods or processes.
...,...
555,Analyze data to inform operational decisions o...
556,Confer with organizational members to accompli...
557,Supervise employees.
558,Monitor organizational procedures to ensure pr...


In [27]:
print(len(dwa['dwa_title'].tolist()))

560


In [22]:
import re, unicodedata

# --- Regex precompiles ---
RE_ZERO_WIDTH   = re.compile(r'[\u200B-\u200F\u202A-\u202E\u2060\uFEFF]')
RE_SPACES       = re.compile(r'\s+')
RE_TRAILING_DOT = re.compile(r'[.\s]+$')

# Clase de comillas (ASCII + tipográficas + angulares)
QUOTE_UTF8_CHARS = '"\'`“”„‟‘’‚‛«»‹›'
RE_EDGE_QUOTES   = re.compile(rf'^[{re.escape(QUOTE_UTF8_CHARS)}\s]+|[{re.escape(QUOTE_UTF8_CHARS)}\s]+$')

# Mojibake -> UTF-8/ASCII
MOJIBAKE_MAP = {
    "â¢": "•",
    "â": "—",
    "â": "–",
    "â¦": "…",
    "â": "'",
    "â": "'",
    "â": '"',
    "â": '"',
    "â": "",
}

# Enumeraciones iniciales (NO metas comillas aquí)
LEADING_ENUM_PATTERNS = [
    re.compile(r'^\s*\d+\s*\t+\s*'),               # "12\t..."
    re.compile(r'^\s*\d+\s*[\.\)]\s*'),            # "1." / "2)"
    re.compile(r'^\s*[\(\[]\s*\d+\s*[\)\]]\s*'),   # "(1)" / "[2]"
    re.compile(r'^\s*\d+\s*[-–—]\s+'),             # "1 - " / "1 – " / "1 — "
    re.compile(r'^\s*[A-Za-z]\s*[\.\)]\s+'),       # "a) " / "B. "
    re.compile(r'^\s*[ivxlcdmIVXLCDM]+\s*[\.\)]\s+'),  # "IV. " / "i) "
    re.compile(r'^\s*[•\-\*\u2219·–—]\s*\t+\s*'),  # bullets con tab
    re.compile(r'^\s*[•\-\*\u2219·–—]\s+'),        # bullets con espacio
]

def _strip_edge_quotes_loop(s: str) -> str:
    """Recorta comillas/espacios en bordes de forma repetida (izq./der.)."""
    while True:
        new_s = RE_EDGE_QUOTES.sub('', s).strip()
        if new_s == s:
            return s
        s = new_s

def clean_skill(text: str) -> str:
    """
    Devuelve la skill limpia en minúsculas:
      - quita enumeraciones iniciales,
      - recorta comillas de borde (ASCII/UTF-8),
      - corrige mojibake, elimina zero-width/bidi/BOM,
      - quita punto(s) finales,
      - conserva paréntesis internos.
    """
    if text is None:
        return ""
    s = unicodedata.normalize("NFKC", str(text))

    # Mojibake -> UTF8/ASCII
    for bad, good in MOJIBAKE_MAP.items():
        if bad in s:
            s = s.replace(bad, good)

    # Invisibles
    s = RE_ZERO_WIDTH.sub("", s)

    # Enumeraciones al inicio (pelar en capas)
    prev = None
    while s != prev:
        prev = s
        for pat in LEADING_ENUM_PATTERNS:
            s = pat.sub("", s, count=1)

    # Normaliza espacios
    s = RE_SPACES.sub(" ", s).strip()

    # 1ª pasada: recortar comillas de borde
    s = _strip_edge_quotes_loop(s)

    # Quitar punto(s) finales
    s = RE_TRAILING_DOT.sub("", s).strip()

    # 2ª pasada: recortar comillas de borde (para casos como ”.)
    s = _strip_edge_quotes_loop(s)

    # Minúsculas
    return s.lower().strip()

def _t(inp, expected):
    got = clean_skill(inp)
    assert got == expected, f"\nINPUT : {repr(inp)}\nGOT   : {repr(got)}\nEXPECT: {repr(expected)}"

# Comillas (ASCII, tipográficas, angulares) + punto final
_t('"Python Programming"', 'python programming')
_t('“Python Programming”.', 'python programming')
_t("‘Skill’", 'skill')
_t("«Skill»", 'skill')
_t("‹Skill›", 'skill')
_t("`Skill`", 'skill')
_t("' Skill '", 'skill')

# Punto final en skills largas
_t("Long descriptive skill.", "long descriptive skill")
_t("C++.", "c++")

# Mayúsculas/minúsculas
_t("JAVA", "java")
_t("Title Case Mixed", "title case mixed")

# Enumeraciones (número + tab / . / ) / (1) / [2] / 1 - / letra) / romanos / bullets
_t("12\tSystems Design", "systems design")
_t("1.\tSkill", "skill")
_t("(1)\tSkill", "skill")
_t("[2] Skill", "skill")
_t("1 - Skill", "skill")
_t("a)\tSkill", "skill")
_t("IV.\tSkill", "skill")
_t("•\tLeadership", "leadership")
_t("-\tProject Management", "project management")
_t("—\tSystems", "systems")
_t("–\tSystems", "systems")
_t("• bullet style skill", "bullet style skill")
_t("·\tAnother Skill", "another skill")

# Mojibake en medio del texto + zero-width
_t("4Aâs Marketing", "4a's marketing")
_t("Hâow to Design", "how to design")

# Paréntesis preservados
_t("Design (CAD)", "design (cad)")
_t("A/B Testing (advanced)", "a/b testing (advanced)")


In [28]:
temp=dwa['dwa_title'].apply(clean_skill)


display(temp)

0                             update technical knowledge
1       determine operational criteria or specifications
2                            prepare operational reports
3               implement design or process improvements
4                 develop technical methods or processes
                             ...                        
555    analyze data to inform operational decisions o...
556    confer with organizational members to accompli...
557                                  supervise employees
558    monitor organizational procedures to ensure pr...
559    manage control system activities in organizations
Name: dwa_title, Length: 560, dtype: object

In [29]:
conteo = pd.DataFrame(temp.value_counts())

display(conteo)

,count
dwa_title,
design industrial processing systems,11
"recommend technical design or process changes to improve efficiency, quality, or performance",11
program robotic equipment,11
design electromechanical equipment or systems,11
estimate operational costs,10
...,...
monitor performance of organizational members or partners,1
manage inventories of products or organizational resources,1
develop organizational goals or objectives,1


In [30]:
for i in temp:
    print(i) 


update technical knowledge
determine operational criteria or specifications
prepare operational reports
implement design or process improvements
develop technical methods or processes
prepare technical reports for internal use
maintain operational records or records systems
research advanced engineering designs or applications
coordinate activities with suppliers, contractors, clients, or other departments
provide technical guidance to other personnel
conduct quantitative failure analyses of operational data
determine design criteria or specifications
estimate operational costs
determine design criteria or specifications
devise research or testing protocols
evaluate technical data to determine effect on designs or plans
calibrate scientific or technical equipment
create models of engineering designs or methods
design electromechanical equipment or systems
evaluate characteristics of equipment or systems
design electromechanical equipment or systems
test performance of electrical, elect

In [31]:
dwa2 = dwa.copy()
dwa2['dwa_title'] = temp

display(dwa2)

,dwa_title
0,update technical knowledge
1,determine operational criteria or specifications
2,prepare operational reports
3,implement design or process improvements
4,develop technical methods or processes
...,...
555,analyze data to inform operational decisions o...
556,confer with organizational members to accompli...
557,supervise employees
558,monitor organizational procedures to ensure pr...


In [39]:
import csv

formato =[".csv", ".tsv"] 

OUT_DIR2 = Path(path+"/results")
OUT_DIR2.mkdir(parents=True, exist_ok=True)

for i in formato:
    dwa2.to_csv(f"{OUT_DIR2}/dwas{i}", index=False, encoding="utf-8",
                        quoting=csv.QUOTE_MINIMAL, quotechar='"', escapechar='\\')


## Filtrado 8 saved

In [21]:
# Eliminar columnas no deseadas
filtrado8 = df_final[["title", "dwa_title",]]

display(filtrado8)

,title,dwa_title
0,Automotive Engineers,Update technical knowledge.
1,Automotive Engineers,Determine operational criteria or specifications.
2,Automotive Engineers,Prepare operational reports.
3,Automotive Engineers,Implement design or process improvements.
4,Automotive Engineers,Develop technical methods or processes.
...,...,...
555,Quality Control Systems Managers,Analyze data to inform operational decisions o...
556,Quality Control Systems Managers,Confer with organizational members to accompli...
557,Quality Control Systems Managers,Supervise employees.
558,Quality Control Systems Managers,Monitor organizational procedures to ensure pr...


In [40]:
for i in formato:
    filtrado8.to_csv(f"{OUT_DIR2}/filtrado8_results{i}", index=False, encoding="utf-8",
                        quoting=csv.QUOTE_MINIMAL, quotechar='"', escapechar='\\')
